## About YAMNet

[YAMNet](https://github.com/tensorflow/models/tree/master/research/audioset/yamnet) is a pre-trained neural network that employs the [MobileNetV1](https://arxiv.org/abs/1704.04861) depthwise-separable convolution architecture. It can use an audio waveform as input and make independent predictions for each of the 521 audio events from the [AudioSet](http://g.co/audioset) corpus.

Internally, the model extracts "frames" from the audio signal and processes batches of these frames. This version of the model uses frames that are 0.96 second long and extracts one frame every 0.48 seconds .

The model accepts a 1-D float32 Tensor or NumPy array containing a waveform of arbitrary length, represented as single-channel (mono) 16 kHz samples in the range `[-1.0, +1.0]`. This tutorial contains code to help you convert WAV files into the supported format.

The model returns 3 outputs, including the class scores, embeddings (which you will use for transfer learning), and the log mel [spectrogram](https://www.tensorflow.org/tutorials/audio/simple_audio#spectrogram). You can find more details [here](https://tfhub.dev/google/yamnet/1).

One specific use of YAMNet is as a high-level feature extractor - the 1,024-dimensional embedding output. You will use the base (YAMNet) model's input features and feed them into your shallower model consisting of one hidden `tf.keras.layers.Dense` layer. Then, you will train the network on a small amount of data for audio classification _without_ requiring a lot of labeled data and training end-to-end. (This is similar to [transfer learning for image classification with TensorFlow Hub](https://www.tensorflow.org/tutorials/images/transfer_learning_with_hub) for more information.)

First, you will test the model and see the results of classifying audio. You will then construct the data pre-processing pipeline.

### Loading YAMNet from TensorFlow Hub

You are going to use a pre-trained YAMNet from [Tensorflow Hub](https://tfhub.dev/) to extract the embeddings from the sound files.

Loading a model from TensorFlow Hub is straightforward: choose the model, copy its URL, and use the `load` function.

Note: to read the documentation of the model, use the model URL in your browser.

install tensorflow tensorflow-hub librosa soundfile first
"""
YAMNet Environmental Sound Classifier - Local Windows Environment
Adapted from Google Colab notebook for Windows 10 / Python 3.10 / Anaconda
 
Requirements (install in your conda env):
    pip install tensorflow tensorflow-hub librosa soundfile pandas numpy
 
Audio structure expected:
    D:\Complete Participantwise data_26 Jan\P{N}\{Phase}\ucm\snd\*.wav
    Participants: P02 to P18
    Phases: BikeG, BikeU, Tram, WalkG, WalkU
 
Outputs per participant/phase:
    - sound_level  : RMS dB of each 15-second window
    - sound_type   : dominant category code
        {'Wind':0, 'Rural/Natural':1, 'Vehicle':2, 'Urban/Manmade':3, 'Mechanisms':4}
        (always labelled 2 when Vehicle is the dominant class)
"""

In [2]:
!pip install librosa

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ------------------------------ --------- 0.8/1.0 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 4.0 MB/s eta 0:00:00


In [3]:

 
import os
import csv
import io
import math
import glob
from datetime import datetime
 
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import tensorflow as tf
import tensorflow_hub as hub

In [47]:
# ---------------------------------------------------------------------------
# 0.  Configuration
# ---------------------------------------------------------------------------
 
BASE_DIR = r'D:\Complete Participantwise data_26 Jan'
SEGMENT_DURATION = 15          # seconds
SCORE_THRESHOLD  = 0.2         # YAMNet confidence threshold
DESIRED_SR       = 16000       # YAMNet requires 16 kHz mono
 
# Participant timing metadata
# Format: participantID -> phase -> [phase_start, phase_end]
# All timestamps use the unified format: 'M/D/YYYY  H:MM:SS AM/PM'
# 'start_time' is no longer required — it is derived automatically as the
# earliest phase-start time across all phases for each participant.
PARTICIPANT_META = {
    'P02': {
        'BikeG': ['7/11/2025  2:38:10 PM', '7/11/2025  2:58:30 PM'],
        'BikeU': ['7/11/2025  2:02:55 PM', '7/11/2025  2:25:14 PM'],
        'Tram':  ['7/11/2025  4:03:52 PM', '7/11/2025  4:13:50 PM'],
        'WalkG': ['7/11/2025  2:59:42 PM', '7/11/2025  3:34:00 PM'],
        'WalkU': ['7/11/2025  1:17:37 PM', '7/11/2025  1:41:50 PM'],
    },
}
'''
PARTICIPANT_META = {
    'P02': {
        'BikeG': ['7/11/2025  2:38:10 PM', '7/11/2025  2:58:30 PM'],
        'BikeU': ['7/11/2025  2:02:55 PM', '7/11/2025  2:25:14 PM'],
        'Tram':  ['7/11/2025  4:03:52 PM', '7/11/2025  4:13:50 PM'],
        'WalkG': ['7/11/2025  2:59:42 PM', '7/11/2025  3:34:00 PM'],
        'WalkU': ['7/11/2025  1:17:37 PM', '7/11/2025  1:41:50 PM'],
    },
    'P03': {
        'BikeG': ['7/12/2025  1:22:08 PM', '7/12/2025  1:31:26 PM'],
        'BikeU': ['7/12/2025  12:30:56 PM', '7/12/2025  12:55:42 PM'],
        'Tram':  ['7/12/2025  1:42:56 PM', '7/12/2025  1:53:01 PM'],
        'WalkG': ['7/12/2025  1:04:27 PM', '7/12/2025  1:17:54 PM'],
        'WalkU': ['7/12/2025  11:56:41 AM', '7/12/2025  12:23:23 PM'],
    },
    'P04': {
        'BikeG': ['8/25/2025  11:58:40 AM', '8/25/2025  12:11:34 PM'],
        'BikeU': ['8/25/2025  12:40:32 PM', '8/25/2025  1:03:14 PM'],
        'Tram':  ['8/25/2025  12:21:40 PM', '8/25/2025  12:37:31 PM'],
        'WalkG': ['8/25/2025  11:39:29 AM', '8/25/2025  11:56:25 AM'],
        'WalkU': ['8/25/2025  1:10:09 PM', '8/25/2025  1:42:50 PM'],
    },
    'P05': {
        'BikeG': ['7/21/2025  1:32:19 PM', '7/21/2025  1:52:27 PM'],
        'BikeU': ['7/21/2025  2:28:56 PM', '7/21/2025  3:04:13 PM'],
        'Tram':  ['7/21/2025  4:16:59 PM', '7/21/2025  4:27:27 PM'],
        'WalkG': ['7/21/2025  1:52:13 PM', '7/21/2025  2:12:59 PM'],
        'WalkU': ['7/21/2025  3:17:00 PM', '7/21/2025  3:48:11 PM'],
    },
    'P06': {
        'BikeG': ['7/28/2025  1:56:09 PM', '7/28/2025  2:16:16 PM'],
        'BikeU': ['7/28/2025  12:21:09 PM', '7/28/2025  1:05:12 PM'],
        'Tram':  ['7/28/2025  3:09:57 PM', '7/28/2025  3:20:32 PM'],
        'WalkG': ['7/28/2025  2:46:49 PM', '7/28/2025  2:56:44 PM'],
        'WalkU': ['7/28/2025  1:09:31 PM', '7/28/2025  1:37:57 PM'],
    },
    'P07': {
        'BikeG': ['8/1/2025  11:57:24 AM', '8/1/2025  12:24:45 PM'],
        'BikeU': ['8/1/2025  1:35:33 PM', '8/1/2025  2:09:10 PM'],
        'Tram':  ['8/1/2025  2:53:45 PM', '8/1/2025  3:04:32 PM'],
        'WalkG': ['8/1/2025  12:43:49 PM', '8/1/2025  1:11:49 PM'],
        'WalkU': ['8/1/2025  2:14:07 PM', '8/1/2025  2:44:21 PM'],
    },
    'P08': {
        'BikeG': ['8/3/2025  1:44:22 PM', '8/3/2025  1:59:51 PM'],
        'BikeU': ['8/3/2025  11:32:17 AM', '8/3/2025  12:04:30 PM'],
        'Tram':  ['8/3/2025  2:14:52 PM', '8/3/2025  2:23:08 PM'],
        'WalkG': ['8/3/2025  1:11:02 PM', '8/3/2025  1:32:20 PM'],
        'WalkU': ['8/3/2025  12:15:48 PM', '8/3/2025  12:44:23 PM'],
    },
    'P09': {
        'BikeG': ['8/6/2025  11:46:38 AM', '8/6/2025  12:06:12 PM'],
        'BikeU': ['8/6/2025  10:14:13 AM', '8/6/2025  10:40:04 AM'],
        'Tram':  ['8/6/2025  12:08:12 PM', '8/6/2025  12:17:05 PM'],
        'WalkG': ['8/6/2025  11:23:44 AM', '8/6/2025  11:46:46 AM'],
        'WalkU': ['8/6/2025  10:43:24 AM', '8/6/2025  11:17:07 AM'],
    },
    'P10': {
        'BikeG': ['8/12/2025  1:13:54 PM', '8/12/2025  1:25:37 PM'],
        'BikeU': ['8/12/2025  11:47:00 AM', '8/12/2025  12:12:45 PM'],
        'Tram':  ['8/12/2025  1:32:00 PM', '8/12/2025  1:42:04 PM'],
        'WalkG': ['8/12/2025  12:56:52 PM', '8/12/2025  1:12:05 PM'],
        'WalkU': ['8/12/2025  12:16:38 PM', '8/12/2025  12:46:18 PM'],
    },
    'P11': {
        'BikeG': ['8/9/2025  12:03:26 PM', '8/9/2025  12:15:19 PM'],
        'BikeU': ['8/9/2025  1:18:58 PM', '8/9/2025  1:41:44 PM'],
        'Tram':  ['8/9/2025  12:24:56 PM', '8/9/2025  12:34:54 PM'],
        'WalkG': ['8/9/2025  11:41:00 AM', '8/9/2025  12:01:27 PM'],
        'WalkU': ['8/9/2025  12:43:32 PM', '8/9/2025  1:13:18 PM'],
    },
    'P12': {
        'BikeG': ['8/10/2025  12:02:27 PM', '8/10/2025  12:22:18 PM'],
        'BikeU': ['8/10/2025  12:46:15 PM', '8/10/2025  1:08:45 PM'],
        'Tram':  ['8/10/2025  12:24:57 PM', '8/10/2025  12:34:53 PM'],
        'WalkG': ['8/10/2025  11:49:00 AM', '8/10/2025  12:09:00 PM'],
        'WalkU': ['8/10/2025  1:12:55 PM', '8/10/2025  1:40:52 PM'],
    },
    'P13': {
        'BikeG': ['8/13/2025  12:14:34 PM', '8/13/2025  12:23:23 PM'],
        'BikeU': ['8/13/2025  12:48:40 PM', '8/13/2025  1:10:29 PM'],
        'Tram':  ['8/13/2025  12:33:51 PM', '8/13/2025  12:41:15 PM'],
        'WalkG': ['8/13/2025  11:50:55 AM', '8/13/2025  12:10:46 PM'],
        'WalkU': ['8/13/2025  1:16:25 PM', '8/13/2025  1:47:23 PM'],
    },
    'P14': {
        'BikeG': ['8/14/2025  1:33:47 PM', '8/14/2025  1:53:52 PM'],
        'BikeU': ['8/14/2025  11:59:00 AM', '8/14/2025  12:19:34 PM'],
        'Tram':  ['8/14/2025  1:55:01 PM', '8/14/2025  2:03:31 PM'],
        'WalkG': ['8/14/2025  1:14:44 PM', '8/14/2025  1:30:46 PM'],
        'WalkU': ['8/14/2025  12:23:46 PM', '8/14/2025  12:57:02 PM'],
    },
    'P15': {
        'BikeG': ['8/15/2025  12:11:21 PM', '8/15/2025  12:31:52 PM'],
        'BikeU': ['8/15/2025  1:30:59 PM', '8/15/2025  1:49:54 PM'],
        'Tram':  ['8/15/2025  12:36:37 PM', '8/15/2025  12:46:06 PM'],
        'WalkG': ['8/15/2025  11:49:59 AM', '8/15/2025  12:08:37 PM'],
        'WalkU': ['8/15/2025  12:54:16 PM', '8/15/2025  1:26:31 PM'],
    },
    'P16': {
        'BikeG': ['8/17/2025  12:43:00 PM', '8/17/2025  1:07:35 PM'],
        'BikeU': ['8/17/2025  1:28:41 PM', '8/17/2025  1:47:16 PM'],
        'Tram':  ['8/17/2025  1:11:50 PM', '8/17/2025  1:21:32 PM'],
        'WalkG': ['8/17/2025  12:21:41 PM', '8/17/2025  12:42:05 PM'],
        'WalkU': ['8/17/2025  1:49:30 PM', '8/17/2025  2:17:13 PM'],
    },
    'P17': {
        'BikeG': ['8/26/2025  11:54:01 AM', '8/26/2025  12:15:38 PM'],
        'BikeU': ['8/26/2025  1:10:27 PM', '8/26/2025  1:33:01 PM'],
        'Tram':  ['8/26/2025  12:20:05 PM', '8/26/2025  12:31:32 PM'],
        'WalkG': ['8/26/2025  11:36:58 AM', '8/26/2025  11:51:20 AM'],
        'WalkU': ['8/26/2025  12:35:58 PM', '8/26/2025  1:07:42 PM'],
    },
    'P18': {
        'BikeG': ['8/27/2025  12:59:13 PM', '8/27/2025  1:17:36 PM'],
        'BikeU': ['8/27/2025  11:40:04 AM', '8/27/2025  11:59:00 AM'],
        'Tram':  ['8/27/2025  1:19:50 PM', '8/27/2025  1:33:23 PM'],
        'WalkG': ['8/27/2025  12:38:01 PM', '8/27/2025  12:58:13 PM'],
        'WalkU': ['8/27/2025  12:00:37 PM', '8/27/2025  12:25:58 PM'],
    },
}
'''
# Category definitions  (YAMNet AudioSet class labels)
TARGET_CATEGORIES = {
    "Wind": [
        "Wind", "Wind noise (microphone)", "Howl", "Whoosh, swoosh, swish",
        "Gust of wind", "Blowing (wind)", "Storm", "Hurricane", "Tornado"
    ],
    "Rural/Natural": [
        "Outside, rural or natural", "Forest", "Bird",
        "Bird vocalization, bird call, bird song",
        "Chirp, tweet", "Coo", "Crow", "Insect", "Cricket", "Bee, wasp, etc.",
        "Frog", "Nature sounds", "Stream", "Water", "Rain", "Thunderstorm",
        "Thunder", "Ocean", "Waves, surf", "Animal",
        "Livestock, farm animals, working animals",
        "Horse", "Cattle, bovinae", "Sheep", "Goat", "Pig",
        "Chicken, rooster", "Dog", "Cat", "Wild animals", "Wolf",
        "Bird flight, flapping wings"
    ],
    "Vehicle": [
        "Vehicle", "Motor vehicle (road)", "Car", "Car passing by", "Truck",
        "Bus", "Motorcycle", "Emergency vehicle", "Siren",
        "Police car (siren)", "Ambulance (siren)",
        "Fire engine, fire truck (siren)", "Aircraft", "Airplane",
        "Helicopter", "Train", "Rail transport", "Train horn",
        "Subway, metro, underground", "Boat, Water vehicle", "Ship",
        "Motorboat, speedboat", "Engine", "Engine starting", "Idling",
        "Accelerating, revving, vroom", "Car alarm",
        "Vehicle horn, car horn, honking"
    ],
    "Urban/Manmade_neutral": [
        "Outside, urban or manmade", "City", "Urban",
        "Footsteps", "Walking", "Running", "Skateboard", "Bicycle",
    ],
    "Urban/Manmade_positive": [
        "Crowd", "Cheering", "Applause",
        "Chatter", "Conversation", "Speech", "Human voice",
        "Street music", "Busker", "Music",
    ],
    "Urban/Manmade_negative": [
        "Traffic noise, roadway noise", "Construction site", "Jackhammer",
        "Bulldozer", "Excavator", "Drilling", "Hammering", "Sawing",
        "Power tool", "Chainsaw", "Siren", "Emergency vehicle",
        "Industrial noise", "Factory", "Machinery", "Air conditioning"
    ],
    "Mechanisms": [
        "Mechanisms", "Tools", "Hammer", "Hammering", "Wood", "Sawing",
        "Filing (rasp)", "Sanding", "Power tool", "Drill", "Jackhammer",
        "Chainsaw", "Medium engine (mid frequency)",
        "Light engine (high frequency)", "Heavy engine (low frequency)",
        "Engine knocking", "Engine starting", "Idling",
        "Accelerating, revving, vroom", "Door", "Sliding door",
        "Slam", "Knock", "Tap", "Squeak", "Creak", "Ratchet, pawl",
        "Clock", "Tick", "Ticking", "Mechanical fan", "Air conditioning",
        "Microwave oven", "Blender", "Water tap, faucet", "Flush",
        "Zipper (clothing)", "Keys jangling", "Coin (dropping)",
        "Scissors", "Electric shaver, electric razor", "Lawn mower",
        "Printing", "Typewriter", "Keyboard (typing)"
    ],
}
 # Default fallback when no category exceeds SCORE_THRESHOLD
FALLBACK_CATEGORY = 'Unidentify'
# Numeric codes required in the output
CATEGORY_CODES = {
    'Unidentify':0,
    'Wind':          1,
    'Rural/Natural': 2,
    'Vehicle':       3,
    'Urban/Manmade_neutral': 4,
    'Urban/Manmade_positive': 5,
    'Urban/Manmade_negative': 6,
    'Mechanisms':    7,
}
 
PHASES = ['BikeG', 'BikeU', 'Tram', 'WalkG', 'WalkU']
 

### Load the class mapping

It's important to load the class names that YAMNet is able to recognize. The mapping file is present at `yamnet_model.class_map_path()` in the CSV format.

With the model loaded, you can follow the [YAMNet basic usage tutorial](https://www.tensorflow.org/hub/tutorials/yamnet) and download a sample WAV file to run the inference.


You will need a function to load audio files, which will also be used later when working with the training data. (Learn more about reading audio files and their labels in [Simple audio recognition](https://www.tensorflow.org/tutorials/audio/simple_audio#reading_audio_files_and_their_labels).

Note: The returned `wav_data` from `load_wav_16k_mono` is already normalized to values in the `[-1.0, 1.0]` range (for more information, go to [YAMNet's documentation on TF Hub](https://tfhub.dev/google/yamnet/1)).

In [48]:
# ---------------------------------------------------------------------------
# 1.  Load YAMNet once
# ---------------------------------------------------------------------------
 
print("Loading YAMNet model from TensorFlow Hub ...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')
print("YAMNet loaded.\n")
 
 
def load_class_names() -> list:
    """Return the 521 AudioSet class name strings used by YAMNet."""
    class_map_path = yamnet_model.class_map_path().numpy().decode('utf-8')
    with tf.io.gfile.GFile(class_map_path, 'r') as f:
        class_map_csv = f.read()
    names = []
    for row in csv.reader(io.StringIO(class_map_csv)):
        if len(row) >= 3 and row[0] != 'index':
            names.append(row[2])
    return names
 
 
CLASS_NAMES = load_class_names()
print(f"Loaded {len(CLASS_NAMES)} YAMNet class names.\n")
 

Loading YAMNet model from TensorFlow Hub ...
YAMNet loaded.

Loaded 521 YAMNet class names.



### About sound level
Negative dBFS values are completely normal and expected — they don't indicate an error.
dBFS (decibels relative to Full Scale) is a logarithmic scale where 0 dBFS is the maximum possible level (a signal at peak amplitude ±1.0). Everything below that maximum is therefore negative:

0 dBFS — signal is at full scale (clipping threshold)
−6 dBFS — fairly loud, about half the peak amplitude
−20 dBFS — moderate level
−40 dBFS — quiet
−60 dBFS — very quiet background
−inf dBFS — complete silence (all zeros)

In [49]:
# ---------------------------------------------------------------------------
# 2.  Audio helpers
# ---------------------------------------------------------------------------
 
def load_audio_mono_16k(file_path: str) -> np.ndarray:
    """Load a WAV, convert to mono float32, resample to 16 kHz, normalise."""
    audio, sr = librosa.load(file_path, sr=None, mono=True, dtype=np.float32)
    if sr != DESIRED_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=DESIRED_SR)
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    return audio
 
 
def rms_db(segment: np.ndarray) -> float:
    """RMS energy in dBFS. Returns -inf for a silent segment."""
    rms = np.sqrt(np.mean(segment ** 2))
    if rms == 0:
        return -np.inf
    return 20.0 * np.log10(rms)

### Run inference

YAMNet provides frame-level class-scores (i.e., 521 scores for every frame). In order to determine clip-level predictions, the scores can be aggregated per-class across frames (e.g., using mean or max aggregation). This is done below by `scores_np.mean(axis=0)`. Finally, to find the top-scored class at the clip-level, you take the maximum of the 521 aggregated scores.


In [50]:
# ---------------------------------------------------------------------------
# 3.  YAMNet classification
# ---------------------------------------------------------------------------
 
def classify_segment(segment: np.ndarray) -> str:
    """
    Run YAMNet on one segment and return the dominant TARGET_CATEGORIES key.
    Falls back to FALLBACK_CATEGORY ('Unidentify', code 7) when no score exceeds SCORE_THRESHOLD.
    """
    scores, _, _ = yamnet_model(tf.constant(segment, dtype=tf.float32))
    class_scores  = tf.reduce_mean(scores, axis=0).numpy()  # shape (521,)
 
    best_category = FALLBACK_CATEGORY
    best_score    = -1.0
 
    for category, keywords in TARGET_CATEGORIES.items():
        cat_max = 0.0
        for idx, class_name in enumerate(CLASS_NAMES):
            score = class_scores[idx]
            for kw in keywords:
                if kw.lower() in class_name.lower() or class_name.lower() in kw.lower():
                    if score > cat_max:
                        cat_max = score
                    break
        if cat_max > best_score:
            best_score    = cat_max
            best_category = category
 
    if best_score < SCORE_THRESHOLD:
        best_category = FALLBACK_CATEGORY
 
    return best_category

In [51]:
# ---------------------------------------------------------------------------
# 4.  Discover folders
# ---------------------------------------------------------------------------
 
def discover_wav_files() -> list[tuple[str, str, str]]:
    """
    Walk BASE_DIR and return a sorted list of (participant_id, phase, wav_path)
    for every WAV file found at:
        BASE_DIR / [participantID] / [phaseID] / ucm / snd / *.wav
 
    Any participant folder whose name starts with 'P' and any phase subfolder
    matching a name in PHASES is included automatically -- no hard-coded list
    of participant IDs is required.
    """
    found = []
    for pid_folder in sorted(os.listdir(BASE_DIR)):
        if not pid_folder.upper().startswith('P'):
            continue
        pid_path = os.path.join(BASE_DIR, pid_folder)
        if not os.path.isdir(pid_path):
            continue
        for phase in PHASES:
            snd_dir = os.path.join(pid_path, phase, 'ucm', 'snd')
            if not os.path.isdir(snd_dir):
                continue
            wavs = sorted(glob.glob(os.path.join(snd_dir, '*.wav')))
            if not wavs:
                continue
            if len(wavs) > 1:
                print(f"  [WARNING] {pid_folder}/{phase}: "
                      f"{len(wavs)} WAV files found, using the first one.")
            found.append((pid_folder, phase, wavs[0]))
    return found

In [52]:
# ---------------------------------------------------------------------------
# 5.  Main batch loop
# ---------------------------------------------------------------------------
 
def main():
    wav_list = discover_wav_files()
    if not wav_list:
        print(f"No WAV files found under {BASE_DIR}. Check the path.")
        return None
 
    print(f"Found {len(wav_list)} phase recordings across "
          f"{len(set(p for p,_,_ in wav_list))} participants.\n")
 
    all_results = []
    segment_len = SEGMENT_DURATION * DESIRED_SR
 
    for idx, (pid, phase, wav_path) in enumerate(wav_list, 1):
        print(f"[{idx:>3}/{len(wav_list)}]  {pid}  {phase}")
        print(f"  File: {wav_path}")
 
        # Load the phase audio in full -- no trimming needed
        audio         = load_audio_mono_16k(wav_path)
        total_samples = len(audio)
        duration_s    = total_samples / DESIRED_SR
        n_segments    = math.ceil(total_samples / segment_len)
        print(f"  Duration: {duration_s:.1f} s  ->  {n_segments} segments")
 
        records = []
        for i in range(n_segments):
            s   = i * segment_len
            e   = min(s + segment_len, total_samples)
            seg = audio[s:e]
            if len(seg) < segment_len:                          # pad last segment
                seg = np.pad(seg, (0, segment_len - len(seg)), mode='constant')
 
            level     = rms_db(seg)
            category  = classify_segment(seg)
            type_code = CATEGORY_CODES[category]
 
            records.append({
                'participant':  pid,
                'phase':        phase,
                'segment':      i + 1,
                'start_time_s': i * SEGMENT_DURATION,
                'end_time_s':   min((i + 1) * SEGMENT_DURATION, duration_s),
                'sound_level':  round(level, 2),
                'sound_type':   type_code,
                'category':     category,
            })
 
        df      = pd.DataFrame(records)
        out_csv = os.path.join(os.path.dirname(wav_path),
                               f'{pid}_{phase}_sound_analysis.csv')
        df.to_csv(out_csv, index=False)
        print(f"  Saved: {out_csv}\n")
 
        all_results.append(df)
 
    # -------------------------------------------------------------------------
    # Combined output
    # -------------------------------------------------------------------------
    combined_df  = pd.concat(all_results, ignore_index=True)
    combined_csv = os.path.join(BASE_DIR, 'all_participants_sound_analysis.csv')
    combined_df.to_csv(combined_csv, index=False)
 
    print('=' * 70)
    print(f"Combined CSV: {combined_csv}")
    print(f"Total rows  : {len(combined_df)}")
    print("\nColumn descriptions:")
    print("  sound_level  -- RMS level of the 15-s window in dBFS")
    print("  sound_type   -- 0=Wind  1=Rural/Natural  2=Vehicle  "
          "3=Urban/Manmade_neutral  4=Urban/Manmade_positive  "
          "5=Urban/Manmade_negative  6=Mechanisms  7=Unidentify")
    print("  category     -- human-readable label (for verification)")
    return combined_df

In [52]:
# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
 
if __name__ == '__main__':
    result = main()
    if result is not None:
        print("\nFirst few rows:")
        print(result[['participant', 'phase', 'segment',
                       'start_time_s', 'end_time_s',
                       'sound_level', 'sound_type', 'category']
                     ].head(10).to_string(index=False))

  [WARNING] P1/BikeG: 2 WAV files found, using the first one.
  [WARNING] P1/BikeU: 2 WAV files found, using the first one.
  [WARNING] P1/Tram: 2 WAV files found, using the first one.
  [WARNING] P1/WalkG: 2 WAV files found, using the first one.
  [WARNING] P1/WalkU: 2 WAV files found, using the first one.
  [WARNING] P2/BikeG: 2 WAV files found, using the first one.
  [WARNING] P2/BikeU: 2 WAV files found, using the first one.
  [WARNING] P2/Tram: 2 WAV files found, using the first one.
  [WARNING] P2/WalkG: 2 WAV files found, using the first one.
  [WARNING] P2/WalkU: 2 WAV files found, using the first one.
  [WARNING] P3/BikeG: 2 WAV files found, using the first one.
  [WARNING] P3/BikeU: 2 WAV files found, using the first one.
  [WARNING] P3/Tram: 2 WAV files found, using the first one.
  [WARNING] P3/WalkG: 2 WAV files found, using the first one.
  [WARNING] P3/WalkU: 2 WAV files found, using the first one.
Found 89 phase recordings across 18 participants.

[  1/89]  P1  BikeG
  


KeyboardInterrupt

